# T06 — cell2location: Spatial Deconvolution

**Tool:** `cell2location` (Theis Lab)  
**Datasets:** Mouse brain Visium H&E (from T02) + Allen Brain Atlas scRNA-seq reference  
**Paper:** [Kleshchevnikov et al. 2022, Nature Biotechnology](https://doi.org/10.1038/s41587-021-01139-4)

---

## The problem cell2location solves

Visium spots (~55 µm diameter) typically contain **5–30 cells** — sometimes from multiple cell types. When you cluster a Visium dataset (as in T02), you assign one label per spot. But the spot might be 60% oligodendrocytes and 40% astrocytes.

**Spatial deconvolution** resolves this: given a scRNA-seq reference that tells you what each cell type looks like, estimate the **mixture of cell types within each spot**.

## How cell2location works

cell2location is a two-stage Bayesian model:

```
Stage 1: Estimate reference cell type signatures
  scRNA-seq adata → NB regression per gene per cell type → γ (signature matrix)
  
Stage 2: Decompose Visium spots
  Visium counts ~ Poisson(spots × cell types) with γ as decoder
  → q (cell type abundance per spot)
```

The key insight: it accounts for the fact that Visium spots have **variable total counts** and that cell type signatures from scRNA-seq may not perfectly match the tissue due to technical differences.

| Method | Model | Handles variable spot density |
|--------|-------|-------------------------------|
| RCTD | Ridge regression | No |
| Stereoscope | NB, no spatial | No |
| **cell2location** | Bayesian NB, spatial | Yes |
| DestVI | VAE, spatial | Yes |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import cell2location
import matplotlib.pyplot as plt
from matplotlib import rcParams

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'cell2location {cell2location.__version__}')

## 1. Load the Spatial Dataset

We reuse the mouse brain Visium dataset from T02.

In [ ]:
# Load Visium mouse brain (same dataset as T02)
adata_vis = sq.datasets.visium_hne_adata()
print('Visium:', adata_vis)
print('\nspot count:', adata_vis.n_obs)
print('gene count:', adata_vis.n_vars)

In [ ]:
# Preprocessing for cell2location:
# cell2location requires RAW integer counts — do NOT normalize
# Filter low-quality genes only
sc.pp.filter_genes(adata_vis, min_cells=3)

# Mitochondrial genes are uninformative for deconvolution
adata_vis.var['mt'] = adata_vis.var_names.str.startswith('mt-')  # mouse = lowercase
print('MT genes removed:', adata_vis.var['mt'].sum())
adata_vis = adata_vis[:, ~adata_vis.var['mt']]
print('Visium after filtering:', adata_vis.shape)

## 2. Load the scRNA-seq Reference

For deconvolution we need a scRNA-seq reference with annotated cell types. Here we use a publicly available mouse brain scRNA-seq atlas (Allen Brain Atlas subset via cell2location's datasets module).

In [ ]:
# Download Allen Brain Atlas scRNA-seq reference (mouse cortex/hippocampus)
# ~7k cells, 34 cell types, already annotated
try:
    adata_ref = cell2location.datasets.get_allen_polished()
    print('Loaded Allen Brain Atlas reference:', adata_ref)
except Exception:
    # Fallback: use the built-in snRNA-seq mouse brain reference
    import pooch
    print('Downloading reference...')
    path = pooch.retrieve(
        'https://figshare.com/ndownloader/files/35466196',
        known_hash=None,
        fname='allen_brain_ref.h5ad',
    )
    adata_ref = sc.read_h5ad(path)
    print('Reference loaded:', adata_ref)

print('\nCell type distribution:')
print(adata_ref.obs['cell_type'].value_counts().head(10))

In [ ]:
# Align to shared genes (intersection of scRNA-seq and Visium gene sets)
shared_genes = adata_vis.var_names.intersection(adata_ref.var_names)
print(f'Shared genes: {len(shared_genes)} / Visium: {adata_vis.n_vars} / Ref: {adata_ref.n_vars}')
adata_vis = adata_vis[:, shared_genes].copy()
adata_ref = adata_ref[:, shared_genes].copy()

## 3. Stage 1 — Estimate Reference Cell Type Signatures

Fit a negative binomial regression per cell type to get per-gene expression signatures (γ matrix). This is done on the scRNA-seq reference only.

In [ ]:
# Setup the scRNA-seq reference for NB regression
# labels_key: obs column with cell type annotations
# batch_key: optional, for multi-sample references
cell2location.models.RegressionModel.setup_anndata(
    adata_ref,
    labels_key='cell_type',
)

# Instantiate and train the regression model
mod = cell2location.models.RegressionModel(adata_ref)
print(mod)
mod.train(
    max_epochs=200,
    use_gpu=False,     # set True if GPU available
)

In [ ]:
# Extract the cell type expression signatures
# inf_aver: inferred average expression per cell type per gene
# Shape: n_cell_types × n_genes
adata_ref = mod.export_posterior(
    adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500, 'use_gpu': False}
)

# Get the signature matrix used as input to stage 2
inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][
    [f'means_per_cluster_mu_fg_{c}' for c in adata_ref.uns['mod']['factor_names']]
].T
print('Signature matrix (cell types × genes):', inf_aver.shape)
print('Cell types:', list(inf_aver.index))

## 4. Stage 2 — Deconvolve Visium Spots

In [ ]:
# Setup the Visium data for cell2location
cell2location.models.Cell2location.setup_anndata(
    adata_vis,
    batch_key=None,
)

# N_cells_per_location: expected average cells per Visium spot
# For standard Visium: ~5–15 cells; for Visium HD: ~1–2
# detection_alpha: controls how "strict" spot → cell type mapping is
mod2 = cell2location.models.Cell2location(
    adata_vis,
    cell_state_df=inf_aver,
    N_cells_per_location=10,
    detection_alpha=20,
)
print(mod2)
mod2.train(
    max_epochs=300,
    batch_size=None,        # full data in each epoch (Visium is small)
    train_size=1,
    use_gpu=False,
)

In [ ]:
# Export posterior estimates — q (cell abundance per spot)
adata_vis = mod2.export_posterior(
    adata_vis,
    sample_kwargs={'num_samples': 1000, 'batch_size': mod2.adata.n_obs, 'use_gpu': False}
)

# Add 5% credible interval to obs for visualization
adata_vis.obs[adata_vis.uns['mod']['factor_names']] = (
    adata_vis.obsm['q05_cell_abundance_w_sf']
)
print('Cell type abundances per spot:')
print(adata_vis.obs[adata_vis.uns['mod']['factor_names']].head())

## 5. Visualize Cell Type Distributions on Tissue

In [ ]:
# Spatial scatter plots for each cell type
cell_types = adata_vis.uns['mod']['factor_names']

# Plot top 6 most abundant cell types
top_cts = adata_vis.obs[cell_types].mean().nlargest(6).index.tolist()

for ct in top_cts:
    sq.pl.spatial_scatter(
        adata_vis,
        color=ct,
        title=ct,
        color_map='magma',
        size=1.5,
        figsize=(4, 4),
    )

In [ ]:
# Spatial co-localization of two cell types
# e.g., are oligodendrocytes and astrocytes spatially segregated?
if 'Oligodendrocytes' in cell_types and 'Astrocytes' in cell_types:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, ct in zip(axes, ['Oligodendrocytes', 'Astrocytes']):
        sc.pl.spatial(adata_vis, color=ct, color_map='magma', ax=ax, show=False)
        ax.set_title(ct)
    plt.tight_layout()
    plt.show()

In [ ]:
# Spot composition: pie-chart style summary
import matplotlib.pyplot as plt

mean_abundances = adata_vis.obs[cell_types].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(mean_abundances.index[:10][::-1], mean_abundances.values[:10][::-1])
ax.set_xlabel('Mean estimated cells per spot')
ax.set_title('Cell type composition (averaged across all spots)')
plt.tight_layout()
plt.show()

---
## Summary

```python
# cell2location two-stage workflow

# Stage 1: scRNA-seq reference → cell type signatures
cell2location.models.RegressionModel.setup_anndata(adata_ref, labels_key='cell_type')
mod1 = cell2location.models.RegressionModel(adata_ref)
mod1.train(max_epochs=200)
adata_ref = mod1.export_posterior(adata_ref, ...)
inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][...].T

# Stage 2: Visium spots → cell type abundances
cell2location.models.Cell2location.setup_anndata(adata_vis)
mod2 = cell2location.models.Cell2location(adata_vis, cell_state_df=inf_aver,
                                          N_cells_per_location=10)
mod2.train(max_epochs=300)
adata_vis = mod2.export_posterior(adata_vis, ...)
# adata_vis.obs now has per-spot cell type abundance columns
```

| Parameter | Default | Notes |
|-----------|---------|-------|
| `N_cells_per_location` | — | Estimated cells per Visium spot (~10 for standard, ~1 for HD) |
| `detection_alpha` | 20 | Higher = stricter deconvolution |
| `max_epochs` stage 1 | 200 | More = better signatures |
| `max_epochs` stage 2 | 300 | More = tighter posteriors |

**Next:** T07 — Moscot for optimal transport trajectories.